In [32]:
!pip install requests beautifulsoup4 pandas -q

import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [33]:
HEADERS = {"User-Agent": "Mozilla/5.0"}
PRINT_ALL_HTML = False

In [34]:
# ==================================
# PART 1: Books (books.toscrape.com)
# ==================================
books_data = []

for i in range(1, 51):
    url = f"https://books.toscrape.com/catalogue/page-{i}.html"

    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        response.encoding = "utf-8"
        html = response.text
    except requests.RequestException as error:
        print("Book page error:", url, error)
        continue

    # Q1: Display HTML source
    if PRINT_ALL_HTML or i == 1:
        print(f"\nBOOK PAGE {i}: {url}")
        print(html)

    # Q2: Extract data
    soup = BeautifulSoup(html, "html.parser")
    books = soup.select("article.product_pod")

    for book in books:
        link = book.select_one("h3 a")
        books_data.append({
            "title": link["title"],
            "price": book.select_one(".price_color").get_text(strip=True),
            "availability": book.select_one(".availability").get_text(strip=True),
            "rating": book.select_one(".star-rating")["class"][1],
            "book_url": urljoin(url, link["href"]),
            "page": i
        })

    print(f"Page {i}: {len(books)} books | Total: {len(books_data)}")
    time.sleep(0.3)

# Q3: Save to CSV
books_df = pd.DataFrame(books_data)
books_df.to_csv("books.csv", index=False, encoding="utf-8-sig")
print("\nSaved books.csv")
print(books_df.head())


BOOK PAGE 1: https://books.toscrape.com/catalogue/page-1.html


<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
        <meta name="created" content="24th Jun 2016 09:30" />
        <meta name="description" content="" />
        <meta name="viewport" content="width=device-width" />
        <meta name="robots" content="NOARCHIVE,NOCACHE" />

        <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
        <!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->

        


In [35]:
# ==================================
# PART 2: Quotes (quotes.toscrape.com)
# ==================================
quotes_data = []
visited = set()
next_url = "https://quotes.toscrape.com/"
page_no = 0

while next_url and next_url not in visited:
    url = next_url
    visited.add(url)
    page_no += 1

    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        response.encoding = "utf-8"
        html = response.text
    except requests.RequestException as error:
        print("Quote page error:", url, error)
        break

    # Q1: Display HTML source
    if PRINT_ALL_HTML or page_no == 1:
        print(f"\nQUOTE PAGE {page_no}: {url}")
        print(html)

    # Q2: Extract data
    soup = BeautifulSoup(html, "html.parser")
    quotes = soup.select("div.quote")

    for quote in quotes:
        quotes_data.append({
            "quote": quote.select_one(".text").get_text(strip=True),
            "author": quote.select_one(".author").get_text(strip=True),
            "tags": ", ".join(t.get_text(strip=True) for t in quote.select(".tags a.tag")),
            "page_url": url
        })

    print(f"Page {page_no}: {len(quotes)} quotes | Total: {len(quotes_data)}")

    next_button = soup.select_one("li.next a")
    next_url = urljoin(url, next_button["href"]) if next_button else None
    time.sleep(0.3)



QUOTE PAGE 1: https://quotes.toscrape.com/
<!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
    
</head>
<body>
    <div class="container">
        <div class="row header-box">
            <div class="col-md-8">
                <h1>
                    <a href="/" style="text-decoration: none">Quotes to Scrape</a>
                </h1>
            </div>
            <div class="col-md-4">
                <p>
                
                    <a href="/login">Login</a>
                
                </p>
            </div>
        </div>
    

<div class="row">
    <div class="col-md-8">

    <div class="quote" itemscope itemtype="http://schema.org/CreativeWork">
        <span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.

In [36]:
# Q3: Save to CSV
quotes_df = pd.DataFrame(quotes_data).drop_duplicates(subset=["quote"])
quotes_df.to_csv("quotes.csv", index=False, encoding="utf-8-sig")
print("\nSaved quotes.csv")
print(quotes_df.head())


Saved quotes.csv
                                               quote           author  \
0  “The world as we have created it is a process ...  Albert Einstein   
1  “It is our choices, Harry, that show what we t...     J.K. Rowling   
2  “There are only two ways to live your life. On...  Albert Einstein   
3  “The person, be it gentleman or lady, who has ...      Jane Austen   
4  “Imperfection is beauty, madness is genius and...   Marilyn Monroe   

                                           tags                      page_url  
0        change, deep-thoughts, thinking, world  https://quotes.toscrape.com/  
1                            abilities, choices  https://quotes.toscrape.com/  
2  inspirational, life, live, miracle, miracles  https://quotes.toscrape.com/  
3              aliteracy, books, classic, humor  https://quotes.toscrape.com/  
4                    be-yourself, inspirational  https://quotes.toscrape.com/  


In [37]:
# ==================================
# Q4: Verify 1000 rows
# ==================================
books_rows = len(pd.read_csv("books.csv"))
quotes_rows = len(pd.read_csv("quotes.csv"))

print("\n========== FINAL CHECK ==========")
print("Books rows  :", books_rows, "| >= 1000 ?", books_rows >= 1000)
print("Quotes rows :", quotes_rows)


========== FINAL CHECK ==========
Books rows  : 1000 | >= 1000 ? True
Quotes rows : 100
